# ProtT5 Glycosylation Prediction (Clean Pipeline)

This notebook builds a clean, reproducible protein-level classifier to predict whether a protein is glycosylated, using **ProtT5-XL-UniRef50** (1024-dim) pooled embeddings.

## What this notebook does
- Loads UniProt data and defines binary labels using the Glycosylation column (notna = positive).
- Loads precomputed ProtT5 embeddings (max pool and attention pool, 1024-dim).
- Trains and compares Logistic Regression and MLP models.
- Evaluates ROC-AUC, PR-AUC, precision, recall, F1, and confusion matrices.
- Saves clean plots and summary tables to `results_glycosylation_ProtT5/`.
- Also produces PCA / t-SNE / UMAP embedding projections.
- Generates a self-contained HTML interpretation report.

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
PROJECT_DIR = Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results_glycosylation_ProtT5"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Locate UniProt TSV (local all_uniref or NFS)
candidate_data_dirs = [
    (PROJECT_DIR / ".." / "all_uniref").resolve(),
    (PROJECT_DIR / ".." / ".." / "all_uniref").resolve(),
    Path("/nfs/turbo/umms-mcieslik/saishyam/Protein_dynamics/all_uniref"),
]
DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), candidate_data_dirs[0])
UNIPROT_FILE = DATA_DIR / "uniprotkb_AND_reviewed_true_AND_model_o_2026_01_16.tsv"

# Locate ProtT5 pooled embeddings
candidate_embed_files = [
    DATA_DIR / "protT5_embedding_cache" / "pooled_embeddings.npz",
    Path("/nfs/turbo/umms-mcieslik/saishyam/Protein_dynamics/all_uniref/protT5_embedding_cache/pooled_embeddings.npz"),
]
EMBED_FILE = next((p for p in candidate_embed_files if p.exists()), candidate_embed_files[0])

print(f"Project directory : {PROJECT_DIR}")
print(f"Data directory    : {DATA_DIR}")
print(f"Results directory : {RESULTS_DIR}")
print(f"UniProt file      : {UNIPROT_FILE}  (exists: {UNIPROT_FILE.exists()})")
print(f"Embedding file    : {EMBED_FILE}  (exists: {EMBED_FILE.exists()})")

In [ ]:
all_uniref = pd.read_csv(UNIPROT_FILE, sep="\t")
print(f"Loaded {len(all_uniref):,} proteins")
all_uniref.head()

In [ ]:
def build_clean_label_sets(df: pd.DataFrame) -> tuple:
    # Positive: Glycosylation column is populated
    glyco_mask = df["Glycosylation"].notna()

    glyco  = df[glyco_mask].copy()
    nglyco = df[~glyco_mask].copy()

    cols = ["Entry", "Protein names", "Sequence"]
    rename_map = {
        "Entry": "protein_id",
        "Protein names": "protein_name",
        "Sequence": "sequence",
    }

    glyco  = glyco.loc[:, cols].rename(columns=rename_map)
    nglyco = nglyco.loc[:, cols].rename(columns=rename_map)

    glyco  = glyco.drop_duplicates(subset="protein_name", keep="first")
    nglyco = nglyco.drop_duplicates(subset="protein_name", keep="first")

    overlap = set(glyco["protein_name"]) & set(nglyco["protein_name"])
    glyco  = glyco[~glyco["protein_name"].isin(overlap)]
    nglyco = nglyco[~nglyco["protein_name"].isin(overlap)]

    glyco  = glyco.reset_index(drop=True)
    nglyco = nglyco.reset_index(drop=True)

    assert glyco["protein_name"].is_unique
    assert nglyco["protein_name"].is_unique
    assert set(glyco["protein_name"]).isdisjoint(set(nglyco["protein_name"]))

    return glyco, nglyco

glyco_df, nglyco_df = build_clean_label_sets(all_uniref)

class_counts = pd.DataFrame({
    "Class": ["Glycosylated", "Not glycosylated"],
    "Count": [len(glyco_df), len(nglyco_df)]
})
class_counts["Fraction"] = class_counts["Count"] / class_counts["Count"].sum()
print(class_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=class_counts, x="Class", y="Count", palette=["#d62728", "#1f77b4"], ax=ax)
ax.set_title("Class Distribution — ProtT5 Glycosylation")
ax.set_xlabel("")
ax.set_ylabel("Number of proteins")
for i, row in class_counts.iterrows():
    ax.text(i, row["Count"] * 1.01, f"{row['Count']} ({row['Fraction']:.1%})",
            ha="center", va="bottom", fontsize=11)
plt.tight_layout()
class_dist_path = RESULTS_DIR / "class_distribution.png"
plt.savefig(class_dist_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {class_dist_path}")

In [ ]:
print(f"Loading ProtT5 embeddings from: {EMBED_FILE}")
t0 = time.perf_counter()
cache = np.load(EMBED_FILE, allow_pickle=True)
max_pool  = cache["max"].item()
attn_pool = cache["attn"].item()
print(f"Loaded in {time.perf_counter()-t0:.1f}s")

def collect_embeddings(df: pd.DataFrame, pool_dict: dict, label: str) -> np.ndarray:
    vectors = []
    missing = []
    for pid in df["protein_id"]:
        vec = pool_dict.get(pid)
        if vec is None:
            missing.append(pid)
        else:
            vectors.append(vec)
    if not vectors:
        raise ValueError(f"No embeddings found for class: {label}")
    if missing:
        print(f"[{label}] Missing embeddings: {len(missing)}")
    return np.stack(vectors)

X_glyco_max   = collect_embeddings(glyco_df,  max_pool,  "glycosylated|max")
X_nglyco_max  = collect_embeddings(nglyco_df, max_pool,  "not_glycosylated|max")
X_glyco_attn  = collect_embeddings(glyco_df,  attn_pool, "glycosylated|attn")
X_nglyco_attn = collect_embeddings(nglyco_df, attn_pool, "not_glycosylated|attn")

print("Max pool shapes :", X_glyco_max.shape,  X_nglyco_max.shape)
print("Attn pool shapes:", X_glyco_attn.shape, X_nglyco_attn.shape)
assert X_glyco_max.shape[1] == 1024, f"Expected 1024-dim ProtT5 embeddings, got {X_glyco_max.shape[1]}"

In [ ]:
def make_split(pos: np.ndarray, neg: np.ndarray):
    X = np.vstack([pos, neg]).astype(np.float32)
    y = np.array([1] * len(pos) + [0] * len(neg), dtype=np.int64)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)
    return X_train, X_test, y_train, y_test

split_data = {
    "max":  make_split(X_glyco_max,  X_nglyco_max),
    "attn": make_split(X_glyco_attn, X_nglyco_attn),
}

def evaluate_binary(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "roc_auc":   roc_auc_score(y_true, y_prob),
        "pr_auc":    average_precision_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

results = []
curve_data = {}

t0 = time.perf_counter()
for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    clf = LogisticRegression(
        max_iter=5000, class_weight="balanced", n_jobs=-1, random_state=SEED
    )
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]

    metrics = evaluate_binary(yte, prob)
    metrics.update({
        "model": "logistic_regression", "pooling": pooling_name,
        "n_test": int(len(yte)), "positive_rate_test": float(yte.mean()),
    })
    results.append(metrics)

    fpr, tpr, _ = roc_curve(yte, prob)
    prec, rec, _ = precision_recall_curve(yte, prob)
    curve_data[("logistic_regression", pooling_name)] = {
        "y_true": yte, "y_prob": prob,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }
print(f"Logistic regression complete in {time.perf_counter()-t0:.1f}s")

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def train_mlp(Xtr, ytr, Xte, yte, epochs=30, batch_size=64, lr=1e-3):
    Xtr_t = torch.from_numpy(Xtr)
    ytr_t = torch.from_numpy(ytr.astype(np.float32))
    Xte_t = torch.from_numpy(Xte)

    train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    pos = ytr.sum()
    neg = len(ytr) - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=device)

    model = MLP(Xtr.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    model.train()
    for _ in range(epochs):
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        losses.append(running / len(train_loader))

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t.to(device))).cpu().numpy()

    return probs, losses

training_loss = {}
t0 = time.perf_counter()
for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    probs, losses = train_mlp(Xtr, ytr, Xte, yte)
    metrics = evaluate_binary(yte, probs)
    metrics.update({
        "model": "mlp", "pooling": pooling_name,
        "n_test": int(len(yte)), "positive_rate_test": float(yte.mean()),
    })
    results.append(metrics)

    fpr, tpr, _ = roc_curve(yte, probs)
    prec, rec, _ = precision_recall_curve(yte, probs)
    curve_data[("mlp", pooling_name)] = {
        "y_true": yte, "y_prob": probs,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }
    training_loss[pooling_name] = losses
print(f"MLP training complete in {time.perf_counter()-t0:.1f}s")

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[[
    "model", "pooling", "roc_auc", "pr_auc",
    "precision", "recall", "f1",
    "tp", "fp", "tn", "fn",
    "n_test", "positive_rate_test",
]]
results_df = results_df.sort_values(["pr_auc", "roc_auc"], ascending=False).reset_index(drop=True)

csv_path = RESULTS_DIR / "summary_metrics.csv"
md_path  = RESULTS_DIR / "summary_metrics.md"
results_df.to_csv(csv_path, index=False)
header    = "| " + " | ".join(results_df.columns) + " |"
separator = "|" + "|".join(["---"] * len(results_df.columns)) + "|"
rows      = ["| " + " | ".join(map(str, row)) + " |" for row in results_df.to_numpy()]
md_path.write_text("\n".join([header, separator] + rows))

display(results_df.style.format({
    "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "precision": "{:.4f}", "recall": "{:.4f}",
    "f1": "{:.4f}", "positive_rate_test": "{:.4f}",
}))
print(f"Saved: {csv_path}")
print(f"Saved: {md_path}")

In [ ]:
palette = {
    ("logistic_regression", "max"):  "#1f77b4",
    ("logistic_regression", "attn"): "#ff7f0e",
    ("mlp", "max"):                  "#2ca02c",
    ("mlp", "attn"):                 "#d62728",
}

fig, ax = plt.subplots(figsize=(8.5, 6))
for key, data in curve_data.items():
    model, pooling = key
    label = f"{model} | {pooling} (AUC={roc_auc_score(data['y_true'], data['y_prob']):.3f})"
    ax.plot(data["fpr"], data["tpr"], label=label, color=palette[key], linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5)
ax.set_title("ROC Curves — ProtT5 Glycosylation")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=10, loc="lower right")
plt.tight_layout()
roc_path = RESULTS_DIR / "roc_curves.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 6))
baseline = list(curve_data.values())[0]["y_true"].mean()
for key, data in curve_data.items():
    model, pooling = key
    label = f"{model} | {pooling} (AP={average_precision_score(data['y_true'], data['y_prob']):.3f})"
    ax.plot(data["recall"], data["precision"], label=label, color=palette[key], linewidth=2)
ax.axhline(baseline, linestyle="--", color="gray", linewidth=1.5, label=f"Random baseline={baseline:.3f}")
ax.set_title("Precision-Recall Curves — ProtT5 Glycosylation")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=10, loc="lower left")
plt.tight_layout()
pr_path = RESULTS_DIR / "pr_curves.png"
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, key in zip(axes.ravel(), curve_data.keys()):
    model, pooling = key
    y_true = curve_data[key]["y_true"]
    y_pred = (curve_data[key]["y_prob"] >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"{model} | {pooling}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices — ProtT5 Glycosylation", fontsize=14)
plt.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrices.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.plot(training_loss["max"],  label="MLP | max",  linewidth=2)
ax.plot(training_loss["attn"], label="MLP | attn", linewidth=2)
ax.set_title("MLP Training Loss — ProtT5 Glycosylation")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.legend()
plt.tight_layout()
loss_path = RESULTS_DIR / "mlp_training_loss.png"
plt.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {roc_path}")
print(f"Saved: {pr_path}")
print(f"Saved: {cm_path}")
print(f"Saved: {loss_path}")

In [ ]:
# ============================================================
# PCA + t-SNE + UMAP embedding projections (ProtT5-specific)
# ============================================================
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

X_vis = np.vstack([X_glyco_attn, X_nglyco_attn])
y_vis = np.array(["Glycosylated"] * len(X_glyco_attn) + ["Not glycosylated"] * len(X_nglyco_attn))

# ---- PCA ----
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_vis)

# ---- t-SNE / UMAP (stratified subsample if large) ----
max_tsne_points = 10000
if len(X_vis) > max_tsne_points:
    rng = np.random.default_rng(SEED)
    idx_pos = np.where(y_vis == "Glycosylated")[0]
    idx_neg = np.where(y_vis == "Not glycosylated")[0]
    n_pos = max(1, int(max_tsne_points * len(idx_pos) / len(X_vis)))
    n_neg = max_tsne_points - n_pos
    sample_idx = np.concatenate([
        rng.choice(idx_pos, size=min(n_pos, len(idx_pos)), replace=False),
        rng.choice(idx_neg, size=min(n_neg, len(idx_neg)), replace=False),
    ])
    X_tsne_in = X_vis[sample_idx]
    y_tsne    = y_vis[sample_idx]
    print(f"t-SNE / UMAP on stratified subset: {len(sample_idx)} points")
else:
    X_tsne_in = X_vis
    y_tsne    = y_vis
    print(f"t-SNE / UMAP on full set: {len(X_vis)} points")

t0 = time.perf_counter()
tsne_2d = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED).fit_transform(X_tsne_in)
print(f"t-SNE complete in {time.perf_counter()-t0:.1f}s")

t0 = time.perf_counter()
umap_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(X_tsne_in)
print(f"UMAP complete in {time.perf_counter()-t0:.1f}s")

# ---- Plot all three side by side ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
plot_specs = [
    (axes[0], pca_2d,  y_vis,   "PCA (ProtT5 Attention embeddings)"),
    (axes[1], tsne_2d, y_tsne,  "t-SNE (ProtT5 Attention embeddings)"),
    (axes[2], umap_2d, y_tsne,  "UMAP (ProtT5 Attention embeddings)"),
]
for ax, emb, y_plot, title in plot_specs:
    idx_neg = y_plot == "Not glycosylated"
    idx_pos = y_plot == "Glycosylated"
    ax.scatter(emb[idx_neg, 0], emb[idx_neg, 1], s=14, alpha=0.35, c="#1f77b4", label="Not glycosylated")
    ax.scatter(emb[idx_pos, 0], emb[idx_pos, 1], s=14, alpha=0.55, c="#d62728", label="Glycosylated")
    ax.set_title(title)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
axes[0].legend(loc="best")
plt.suptitle(
    "Dimensionality Reduction — ProtT5 Attention Pool Embeddings\nGlycosylated vs Not Glycosylated",
    fontsize=13
)
plt.tight_layout()
emb_path = RESULTS_DIR / "embedding_projection_pca_tsne_umap.png"
plt.savefig(emb_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {emb_path}")

In [ ]:
best_row = results_df.iloc[0]
summary_lines = [
    "ANALYSIS SUMMARY — ProtT5 Glycosylation",
    "========================================",
    f"Best model by PR-AUC: {best_row['model']} + {best_row['pooling']}",
    f"ROC-AUC  : {best_row['roc_auc']:.4f}",
    f"PR-AUC   : {best_row['pr_auc']:.4f}",
    f"Precision: {best_row['precision']:.4f}",
    f"Recall   : {best_row['recall']:.4f}",
    f"F1       : {best_row['f1']:.4f}",
    "",
    "Interpretation:",
    "- PR-AUC is emphasised because classes may be imbalanced.",
    "- ProtT5 uses 1024-dim embeddings (vs 1280 for ESM2, 768 for Ankh).",
    "- Label: Glycosylation column notna() = positive class.",
    "- UMAP projection included in addition to PCA / t-SNE.",
]

analysis_path = RESULTS_DIR / "analysis_summary.txt"
analysis_path.write_text("\n".join(summary_lines))
print("\n".join(summary_lines))
print(f"\nSaved: {analysis_path}")

In [ ]:
# ============================================================
# Generate comprehensive HTML interpretation report (ProtT5 Glycosylation)
# Opens in any browser; File > Open in Word to save as .docx
# ============================================================
import base64

def img_to_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

# ---- Gather numbers ----
n_glyco   = len(glyco_df)
n_nglyco  = len(nglyco_df)
n_total   = n_glyco + n_nglyco
pos_frac  = n_glyco / n_total
emb_dim   = X_glyco_attn.shape[1]
n_test    = int(results_df['n_test'].iloc[0])

best         = results_df.iloc[0]
best_model   = best['model'].replace('_', ' ').title()
best_pooling = best['pooling'].upper()

# ---- Build metrics table rows ----
def fmt_row(idx, r):
    m  = r['model'].replace('_', ' ').title()
    p  = r['pooling'].upper()
    bg = ' style="background:#fff9e6"' if idx == 0 else ''
    return (
        f'<tr{bg}>'
        f'<td>{m}</td><td>{p}</td>'
        f'<td>{r["roc_auc"]:.4f}</td>'
        f'<td><b>{r["pr_auc"]:.4f}</b></td>'
        f'<td>{r["precision"]:.4f}</td>'
        f'<td>{r["recall"]:.4f}</td>'
        f'<td>{r["f1"]:.4f}</td>'
        f'<td>{r["tp"]}</td><td>{r["fp"]}</td>'
        f'<td>{r["tn"]}</td><td>{r["fn"]}</td>'
        f'</tr>'
    )

table_rows = '\n'.join(fmt_row(i, r) for i, (_, r) in enumerate(results_df.iterrows()))

# ---- Encode figures ----
fig_class = img_to_b64(RESULTS_DIR / 'class_distribution.png')
fig_roc   = img_to_b64(RESULTS_DIR / 'roc_curves.png')
fig_pr    = img_to_b64(RESULTS_DIR / 'pr_curves.png')
fig_cm    = img_to_b64(RESULTS_DIR / 'confusion_matrices.png')
fig_loss  = img_to_b64(RESULTS_DIR / 'mlp_training_loss.png')
fig_emb   = img_to_b64(RESULTS_DIR / 'embedding_projection_pca_tsne_umap.png')

# ---- Compute per-pooling PR-AUC deltas ----
attn_rows    = results_df[results_df['pooling'] == 'attn']
max_rows     = results_df[results_df['pooling'] == 'max']
mlp_attn_pr  = attn_rows[attn_rows['model'] == 'mlp']['pr_auc'].values[0]
mlp_max_pr   = max_rows[max_rows['model']   == 'mlp']['pr_auc'].values[0]
lr_attn_pr   = attn_rows[attn_rows['model'] == 'logistic_regression']['pr_auc'].values[0]
lr_max_pr    = max_rows[max_rows['model']   == 'logistic_regression']['pr_auc'].values[0]

html = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
<meta charset=\"UTF-8\">
<title>ProtT5 Glycosylation Prediction — Interpretation Report</title>
<style>
  body  {{ font-family: 'Segoe UI', Arial, sans-serif; max-width: 1100px; margin: 40px auto; padding: 0 30px; color: #222; line-height: 1.65; }}
  h1    {{ color: #1a3a5c; border-bottom: 3px solid #1a3a5c; padding-bottom: 8px; }}
  h2    {{ color: #2e6da4; margin-top: 42px; }}
  h3    {{ color: #444; margin-top: 24px; }}
  table {{ border-collapse: collapse; width: 100%; margin: 14px 0; font-size: 0.93em; }}
  th    {{ background: #1a3a5c; color: #fff; padding: 8px 10px; text-align: left; }}
  td    {{ padding: 7px 10px; border: 1px solid #ddd; }}
  tr:nth-child(even) td {{ background: #f4f8ff; }}
  .metric-box {{ display: inline-block; background: #f0f6ff; border: 1px solid #b3cde0;
                 border-radius: 6px; padding: 12px 22px; margin: 8px 8px 8px 0;
                 text-align: center; min-width: 130px; vertical-align: top; }}
  .metric-box .val {{ font-size: 1.75em; font-weight: bold; color: #1a3a5c; }}
  .metric-box .lbl {{ font-size: 0.8em; color: #666; margin-top: 4px; }}
  img   {{ max-width: 100%; border: 1px solid #ddd; border-radius: 4px; display: block; margin: 12px auto; }}
  .caption {{ font-size: 0.85em; color: #555; text-align: center; margin-top: 4px; font-style: italic; }}
  .note  {{ background: #f9f9f9; border-left: 4px solid #2e6da4; padding: 10px 16px; margin: 16px 0; font-size: 0.93em; }}
  hr     {{ border: none; border-top: 1px solid #ddd; margin: 32px 0; }}
  code   {{ background: #f4f4f4; padding: 1px 5px; border-radius: 3px; font-size: 0.9em; }}
  .prot5-badge {{ display:inline-block; background:#f3e8ff; border:1px solid #9c27b0;
                  border-radius:4px; padding:2px 8px; font-size:0.85em; color:#6a1b9a; font-weight:bold; }}
</style>
</head>
<body>

<h1>ProtT5 Glycosylation Prediction &mdash; Interpretation Report</h1>
<p style=\"color:#555; font-size:0.9em\">
  Source: <code>ProtT5_glycosylation_clean_pipeline.ipynb</code> &nbsp;|&nbsp;
  Embeddings: <span class=\"prot5-badge\">ProtT5-XL-UniRef50 &bull; 1024-dim</span> &nbsp;|&nbsp;
  Task: binary protein-level glycosylation classification
</p>

<hr>

<h2>1. Dataset</h2>
<p>Proteins from UniProt/Swiss-Prot were split into two mutually exclusive classes using the
<em>Glycosylation</em> column:</p>
<ul>
  <li><b>Positive (glycosylated)</b>: Glycosylation column is populated (notna)</li>
  <li><b>Negative (not glycosylated)</b>: Glycosylation column is empty (NaN)</li>
</ul>
<p>Duplicate protein names were removed within each class and overlaps discarded.
This labelling is identical to the Ankh and ESM2 glycosylation pipelines.</p>

<div>
  <div class=\"metric-box\"><div class=\"val\">{n_glyco:,}</div><div class=\"lbl\">Glycosylated</div></div>
  <div class=\"metric-box\"><div class=\"val\">{n_nglyco:,}</div><div class=\"lbl\">Not glycosylated</div></div>
  <div class=\"metric-box\"><div class=\"val\">{n_total:,}</div><div class=\"lbl\">Total proteins</div></div>
  <div class=\"metric-box\"><div class=\"val\">{pos_frac:.1%}</div><div class=\"lbl\">Positive fraction</div></div>
  <div class=\"metric-box\"><div class=\"val\">{n_test:,}</div><div class=\"lbl\">Test set size (20%)</div></div>
  <div class=\"metric-box\"><div class=\"val\">{emb_dim}</div><div class=\"lbl\">Embedding dim</div></div>
</div>

<img src=\"data:image/png;base64,{fig_class}\" style=\"max-width:500px\">
<p class=\"caption\">Figure 1. Class distribution. Glycosylated proteins represent {pos_frac:.1%} of the dataset.</p>

<div class=\"note\">
  <b>Class imbalance note:</b> PR-AUC (Average Precision) is the primary metric.
  A random classifier achieves PR-AUC &asymp; {pos_frac:.3f}.
  Imbalance is corrected via balanced class weights (LR) and BCEWithLogitsLoss pos_weight (MLP).
</div>

<hr>

<h2>2. Methods</h2>
<h3>2.1 Embeddings</h3>
<p><b>ProtT5-XL-UniRef50</b> ({emb_dim}-dimensional) embeddings were loaded from a pre-built cache.
Two global representations were evaluated: <b>max pooling</b> and <b>attention pooling</b>.
All features z-score normalised (training-set statistics only).</p>
<h3>2.2 Classifiers</h3>
<table>
  <tr><th>Model</th><th>Architecture / Settings</th><th>Imbalance handling</th></tr>
  <tr><td>Logistic Regression</td><td>L2, max_iter=5000, n_jobs=-1</td><td>class_weight=&quot;balanced&quot;</td></tr>
  <tr><td>MLP (PyTorch)</td>
      <td>Linear({emb_dim}&rarr;512)&rarr;ReLU&rarr;Dropout(0.3)&rarr;Linear(512&rarr;256)&rarr;ReLU&rarr;Dropout(0.3)&rarr;Linear(256&rarr;1), Adam lr=1e-3, 30 epochs, batch=64</td>
      <td>BCEWithLogitsLoss pos_weight=neg/pos</td></tr>
</table>
<p>80/20 stratified train/test split, seed 42.</p>

<hr>

<h2>3. Results</h2>
<h3>3.1 Performance Table</h3>
<table>
  <tr><th>Model</th><th>Pooling</th><th>ROC-AUC</th><th>PR-AUC &#9733;</th>
      <th>Precision</th><th>Recall</th><th>F1</th><th>TP</th><th>FP</th><th>TN</th><th>FN</th></tr>
  {table_rows}
</table>
<p class=\"caption\">Random PR-AUC baseline &asymp; {pos_frac:.3f}. Confusion matrix at threshold=0.5.</p>

<h3>3.2 Best Model Summary</h3>
<div>
  <div class=\"metric-box\"><div class=\"val\">{best['roc_auc']:.4f}</div><div class=\"lbl\">ROC-AUC</div></div>
  <div class=\"metric-box\"><div class=\"val\">{best['pr_auc']:.4f}</div><div class=\"lbl\">PR-AUC &#9733;</div></div>
  <div class=\"metric-box\"><div class=\"val\">{best['precision']:.4f}</div><div class=\"lbl\">Precision</div></div>
  <div class=\"metric-box\"><div class=\"val\">{best['recall']:.4f}</div><div class=\"lbl\">Recall</div></div>
  <div class=\"metric-box\"><div class=\"val\">{best['f1']:.4f}</div><div class=\"lbl\">F1</div></div>
</div>
<p>Best: <b>{best_model}</b> + <b>{best_pooling}</b> pooling.</p>

<hr>

<h2>4. Figures</h2>
<h3>4.1 ROC Curves</h3>
<img src=\"data:image/png;base64,{fig_roc}\">
<p class=\"caption\">Figure 2. ROC curves for all four model-pooling combinations.</p>
<h3>4.2 Precision-Recall Curves</h3>
<img src=\"data:image/png;base64,{fig_pr}\">
<p class=\"caption\">Figure 3. PR curves. Dashed line = random baseline ({pos_frac:.3f}).</p>
<h3>4.3 Confusion Matrices</h3>
<img src=\"data:image/png;base64,{fig_cm}\">
<p class=\"caption\">Figure 4. Confusion matrices at decision threshold=0.5.</p>
<h3>4.4 MLP Training Loss</h3>
<img src=\"data:image/png;base64,{fig_loss}\">
<p class=\"caption\">Figure 5. BCE loss over 30 epochs.</p>
<h3>4.5 Embedding Projections (PCA / t-SNE / UMAP)</h3>
<img src=\"data:image/png;base64,{fig_emb}\">
<p class=\"caption\">Figure 6. PCA, t-SNE and UMAP of ProtT5 attention-pool embeddings.</p>

<hr>

<h2>5. Interpretation</h2>
<h3>5.1 MLP vs Logistic Regression</h3>
<p>
  MLP attn PR-AUC: <b>{mlp_attn_pr:.4f}</b> &nbsp;|&nbsp; MLP max PR-AUC: <b>{mlp_max_pr:.4f}</b><br>
  LR attn PR-AUC: <b>{lr_attn_pr:.4f}</b> &nbsp;|&nbsp; LR max PR-AUC: <b>{lr_max_pr:.4f}</b>
</p>
<h3>5.2 Cross-Model Context</h3>
<p>ProtT5-XL uses 1024-dim embeddings, between Ankh (768-dim) and ESM-2 (1280-dim).
Compare <code>results_glycosylation_ProtT5/</code>, <code>results_glycosylation_ESM2/</code>,
and <code>results/</code> (Ankh) for a three-way comparison on identical data.</p>
<h3>5.3 UMAP vs PCA / t-SNE</h3>
<p>UMAP preserves global structure better than t-SNE at scale.
Convergence across all three projections strengthens visual interpretation.
Substantial overlap between classes is expected for protein-level pooled representations.</p>
<h3>5.4 Recommendations</h3>
<ol>
  <li>Residue-level classifier on ProtT5 token embeddings</li>
  <li>5-fold stratified cross-validation instead of single split</li>
  <li>Concatenate max + attention pool (2&times;1024) as joint feature</li>
  <li>Ensemble ProtT5, ESM2 and Ankh embeddings</li>
  <li>Fine-tune ProtT5 end-to-end on the glycosylation task</li>
</ol>

<hr>

<h2>6. Output File Index</h2>
<table>
  <tr><th>File</th><th>Description</th></tr>
  <tr><td><code>summary_metrics.csv</code></td><td>All model metrics (machine-readable)</td></tr>
  <tr><td><code>summary_metrics.md</code></td><td>Markdown table of metrics</td></tr>
  <tr><td><code>analysis_summary.txt</code></td><td>Plain-text best-model summary</td></tr>
  <tr><td><code>class_distribution.png</code></td><td>Bar chart of class balance</td></tr>
  <tr><td><code>roc_curves.png</code></td><td>ROC curves for all configurations</td></tr>
  <tr><td><code>pr_curves.png</code></td><td>Precision-Recall curves</td></tr>
  <tr><td><code>confusion_matrices.png</code></td><td>Four confusion matrices at threshold=0.5</td></tr>
  <tr><td><code>mlp_training_loss.png</code></td><td>BCE loss over 30 epochs</td></tr>
  <tr><td><code>embedding_projection_pca_tsne_umap.png</code></td><td>PCA, t-SNE and UMAP projections</td></tr>
  <tr><td><code>interpretation_report.html</code></td><td>This report</td></tr>
</table>

<hr>
<p style=\"color:#888; font-size:0.82em\">
  <b>To convert to .docx:</b> Open in Microsoft Word (File &rarr; Open &rarr; select .html) &rarr; Save As &rarr; Word Document.<br>
  Or: open in browser &rarr; Print &rarr; Save as PDF.
</p>
</body>
</html>"""

report_path = RESULTS_DIR / 'interpretation_report.html'
report_path.write_text(html, encoding='utf-8')
print(f"Saved: {report_path}")
print("\nTo open as Word .docx:")
print("  Microsoft Word -> File -> Open -> select interpretation_report.html -> Save As -> .docx")